In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [8]:
dataset_path = r"C:\Users\Thimmampalli Asritha\Desktop\Smart Waste\dataset"

In [9]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 1088 images belonging to 3 classes.
Found 271 images belonging to 3 classes.


In [10]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes
])

In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.5965 - loss: 1.1066 - val_accuracy: 0.7601 - val_loss: 0.6522
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 885ms/step - accuracy: 0.7325 - loss: 0.5993 - val_accuracy: 0.5830 - val_loss: 0.8426
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 867ms/step - accuracy: 0.7996 - loss: 0.4935 - val_accuracy: 0.7306 - val_loss: 0.6135
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 881ms/step - accuracy: 0.8594 - loss: 0.3657 - val_accuracy: 0.8007 - val_loss: 0.5048
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 885ms/step - accuracy: 0.8934 - loss: 0.2875 - val_accuracy: 0.7159 - val_loss: 0.6024
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 31s 893ms/step - accuracy: 0.9099 - loss: 0.2390 - val_accuracy: 0.8081 - val_loss: 0.5500
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 875ms/step - accuracy: 0.9412 - loss: 0.1687 - val_accuracy: 0.7454 - val_loss: 0.6334
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 30s 876ms/step - accuracy: 0.9375 - loss: 0.1603 - val_accurac

In [14]:
model.save("waste_classifier.keras")

In [24]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = r"C:\Users\Thimmampalli Asritha\Downloads\i4.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

prediction = model.predict(img_array)

print("Prediction:", prediction)
print("Class:", np.argmax(prediction))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Prediction: [[7.4245375e-01 2.5752273e-01 2.3435981e-05]]
Class: 0


In [25]:
pip install opencv-python


  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)


In [4]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import requests

# ✅ LOAD MODEL (THIS WAS MISSING)
model = load_model("waste_classifier.keras")

# class labels
class_labels = ['metal', 'organic', 'plastic']

# image path
img_path = r"C:\Users\Thimmampalli Asritha\Downloads\i4.jpg"

# preprocess
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# prediction
prediction = model.predict(img_array)
class_index = np.argmax(prediction)
label = class_labels[class_index]

print("Predicted:", label)

# send to backend
url = "http://localhost:5000/data"

data = {
    "wasteType": label,
    "binLevel": 50
}

response = requests.post(url, json=data)

print("Sent to backend:", response.text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
Predicted: metal
Sent to backend: Data stored in MongoDB
